# Generación del modelo

## Pasar los jsonl a datos en csv para poder cargarlos en la base de datos de la API

In [139]:
import pandas as pd
import duckdb
import json

In [140]:
con = duckdb.connect('amazon_etl.db')

### Obtener libros con más de 500 reseñas

In [141]:
con.execute("""
    DROP TABLE IF EXISTS popular_books;
    CREATE TEMP TABLE popular_books AS
    SELECT parent_asin
    FROM read_json_auto('Kindle_Store.jsonl')
    GROUP BY parent_asin
    HAVING COUNT(*) >= 500;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [142]:
total_books = con.execute("""
    SELECT COUNT(*) as total_books
    FROM popular_books
""").df()

total_books

,total_books
0,4601


### Encontrar usuarios con entre 35 y 50 reseñas dentro dichos libros

In [143]:
con.execute("""
    DROP TABLE IF EXISTS active_users;
    CREATE TEMP TABLE active_users AS
    SELECT user_id
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    GROUP BY user_id
    HAVING COUNT(*) BETWEEN 35 AND 50;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [144]:
total_users = con.execute("""
    SELECT COUNT(*) as total_users
    FROM active_users
""").df()

total_users

,total_users
0,2709


### Obtener sólo las reviews de los libros y usuarios filtrados

In [145]:
con.execute("""
    DROP TABLE IF EXISTS final_reviews;
    CREATE TEMP TABLE final_reviews AS
    SELECT user_id, parent_asin as item_id, rating, timestamp
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    AND user_id IN (SELECT user_id FROM active_users);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [146]:
total_reviews = con.execute("""
SELECT COUNT(*) as total_reviews
FROM final_reviews
""").df()

total_reviews

,total_reviews
0,111005


### Obtener metadatos de los libros seleccionados

In [147]:
con.execute("""
    DROP TABLE IF EXISTS final_meta;
    CREATE TEMP TABLE final_meta AS
    SELECT DISTINCT parent_asin as item_id, title, subtitle, description, categories, images, author
    FROM read_json_auto('meta_Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT DISTINCT item_id FROM final_reviews);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [148]:
total_books = con.execute("""
SELECT COUNT(*) as total_books
FROM final_meta
""").df()

total_books

,total_books
0,4533


In [149]:
meta_sample = con.execute("""
    select * from final_meta
    limit 5;
""").df()

print(meta_sample.iloc[0])

item_id                                               B0019LNUVA
title          ROLLING THUNDER: An Historical Novel of War an...
subtitle                                          Kindle Edition
description    [Review, "A taut, exciting tale of good men in...
categories     [Kindle Store, Kindle eBooks, Literature & Fic...
images         [{'large': 'https://m.media-amazon.com/images/...
author         {'avatar': 'https://m.media-amazon.com/images/...
Name: 0, dtype: object


In [150]:
reviews_sample = con.execute("""
    select * from final_reviews
    limit 5;
""").df()

print(reviews_sample.iloc[0])

user_id      AHAMHLUWZTVCHUQLQNDZZAAX4KGA
item_id                        B09J8JSJL5
rating                                5.0
timestamp                   1671887532659
Name: 0, dtype: object


### Exportar a CSV

In [151]:
con.execute("COPY final_reviews TO 'kindle_reviews_seed.csv' (HEADER, DELIMITER ',')")
con.execute("COPY final_meta TO 'kindle_meta_seed.csv' (HEADER, DELIMITER ',')")
con.execute("COPY active_users TO 'active_users.csv' (HEADER, DELIMITER ',')")

In [152]:
meta_df = pd.read_csv('kindle_meta_seed.csv')

In [153]:
print(meta_df.iloc[0]['categories'])

[Kindle Store, Kindle eBooks, Literature & Fiction]


In [154]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        4533
title          4533
subtitle       4155
description    4533
categories     4533
images         4533
author         4338
dtype: int64

### Mantener sólo las categorías relevantes

In [155]:
unwanted_categories = ['Kindle Store', 'Kindle eBooks', 'Kindle Unlimited', 'Kindle Short Reads', 'Kindle eTextbooks', 'Kindle Games & Active Content']
for category in unwanted_categories:
    meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace(category + ',', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace('Kindle eBooks', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.strip() if pd.notnull(x) else x)

In [156]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        0
title          0
subtitle       0
description    0
categories     0
images         0
author         0
dtype: int64

In [157]:
meta_df.head()

,item_id,title,subtitle,description,categories,images,author
0,B0019LNUVA,ROLLING THUNDER: An Historical Novel of War an...,Kindle Edition,"[Review, '""A taut, exciting tale of good men i...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...
1,B076FNQWTF,The Girl Who Lived: A Thrilling Suspense Novel,Kindle Edition,"[Review, 'Praise for Christopher Greyson\'s', ...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...
2,B00FN2KR2G,The Gordonston Ladies Dog Walking Club,Kindle Edition,"[About the Author, 'The life of Duncan Whitehe...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...
3,B00QXUN5Y4,Royal Date (The Royals of Monterra Book 1),Kindle Edition,[],[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...
4,B078JDQ9BH,Defending Allye (Mountain Mercenaries Book 1),Kindle Edition,"[From the Author, 'Mountain MercenariesDefendi...",[ Romance],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...


### Limpiar la columna de imágenes

In [158]:
images = meta_df['images']

In [159]:
meta_df[meta_df['images'].str.len() < 83].__len__()

462

In [160]:
meta_df[meta_df['images'].str.len() > 83].__len__()

316

In [161]:
meta_df[meta_df['images'].str.len() > 83].iloc[0]['images']

"[{'large': 'https://m.media-amazon.com/images/I/51rUUq96UTL._SY346_.jpg', 'variant': MAIN}]"

In [162]:
meta_df[meta_df['images'].str.contains('MAIN') | meta_df['images'].str.len() < 83].__len__()

4533

In [163]:
def get_image_url(images_str):
    if images_str == '[]' or pd.isnull(images_str):
        return ''
    try:
        images_str = images_str.replace("'", '"')
        images_str = images_str.replace("MAIN", '"MAIN"')
        images = json.loads(images_str)
        for image in images:
            if image.get('variant') == 'MAIN':
                return image.get('large', '')
        return images[0].get('large', '') if images else ''
    except json.JSONDecodeError:
        return ''

In [164]:
meta_df['image_url'] = meta_df['images'].apply(get_image_url)

In [165]:
meta_df.head()

,item_id,title,subtitle,description,categories,images,author,image_url
0,B0019LNUVA,ROLLING THUNDER: An Historical Novel of War an...,Kindle Edition,"[Review, '""A taut, exciting tale of good men i...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/519YUR0HNd...
1,B076FNQWTF,The Girl Who Lived: A Thrilling Suspense Novel,Kindle Edition,"[Review, 'Praise for Christopher Greyson\'s', ...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51eAMQG-Gk...
2,B00FN2KR2G,The Gordonston Ladies Dog Walking Club,Kindle Edition,"[About the Author, 'The life of Duncan Whitehe...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51wa4udcDj...
3,B00QXUN5Y4,Royal Date (The Royals of Monterra Book 1),Kindle Edition,[],[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51B5uYHUaa...
4,B078JDQ9BH,Defending Allye (Mountain Mercenaries Book 1),Kindle Edition,"[From the Author, 'Mountain MercenariesDefendi...",[ Romance],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51ET9v-RXO...


### Cortar la descripción 

In [166]:
def truncate_string(s):
    if pd.isnull(s):
        return ''
    return s[:500]

meta_df['description'] = meta_df['description'].apply(truncate_string)

### Limpiar la columna de autor

In [167]:
meta_df['author'].describe()

count                                                  4338
unique                                                 2313
top       {'avatar': 'https://m.media-amazon.com/images/...
freq                                                     29
Name: author, dtype: object

In [168]:
meta_df['author'].str.len().describe()

count     4338.000000
mean      1181.088751
std       1088.767715
min        180.000000
25%        601.000000
50%        877.000000
75%       1325.000000
max      17799.000000
Name: author, dtype: float64

In [169]:
meta_df[meta_df['author'].str.len() > 1000].iloc[700]['author']

"{'avatar': 'https://m.media-amazon.com/images/S/amzn-author-media-prod/p8nppedvfnsjaknd1htkhs445h._SY600_.jpg', 'name': Jillian Dodd, 'about': ['Jillian Dodd® a USA Today and Amazon Top 6 best-selling author. She writes fun binge-able romance series with characters her readers fall in love with—from the boy next door in the That Boy® series to the daughter of a famous actress in The Keatyn Chronicles® to a spy who might save the world in the Spy Girl® series. Her newest series include London Prep, a prep school series about a drama filled three-week exchange, and the Sex in the City-ish chick lit series, Kitty Valentine®.', 'Jillian is married to her college sweetheart, adores writing big fat happily ever afters, wears a lot of pink, buys way too many shoes, loves to travel, and is distracted by anything covered in glitter.', 'Hang out with me:', 'Join my Patreon for early versions of my work: https://www.patreon.com/jilliandodd/', 'Get early access & signed copies: www.jilliandoddsto

In [170]:
meta_df[meta_df['author'].str.len() > 1000].iloc[700]['author'].find(", 'about'")

132

In [171]:
def get_author_name(author_str):
    if pd.isnull(author_str):
        return 'No Disponible'
    start_idx = author_str.find("'name': ")
    if start_idx == -1:
        return 'No Disponible'
    start_idx += len("'name': ")
    end_idx = author_str.find(", 'about'", start_idx)
    return author_str[start_idx:end_idx]

In [172]:
meta_df['author_name'] = meta_df['author'].apply(get_author_name)

In [173]:
get_author_name(meta_df.iloc[0]['author'])

'Mark Berent'

In [174]:
meta_df.iloc[0]['author'].find("'name': ")

112

In [175]:
meta_df.head()

,item_id,title,subtitle,description,categories,images,author,image_url,author_name
0,B0019LNUVA,ROLLING THUNDER: An Historical Novel of War an...,Kindle Edition,"[Review, '""A taut, exciting tale of good men i...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/519YUR0HNd...,Mark Berent
1,B076FNQWTF,The Girl Who Lived: A Thrilling Suspense Novel,Kindle Edition,"[Review, 'Praise for Christopher Greyson\'s', ...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51eAMQG-Gk...,Christopher Greyson
2,B00FN2KR2G,The Gordonston Ladies Dog Walking Club,Kindle Edition,"[About the Author, 'The life of Duncan Whitehe...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51wa4udcDj...,Duncan Whitehead
3,B00QXUN5Y4,Royal Date (The Royals of Monterra Book 1),Kindle Edition,[],[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51B5uYHUaa...,Sariah Wilson
4,B078JDQ9BH,Defending Allye (Mountain Mercenaries Book 1),Kindle Edition,"[From the Author, 'Mountain MercenariesDefendi...",[ Romance],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51ET9v-RXO...,Susan Stoker


### Exportar a csv

In [176]:
books_export = meta_df[['item_id','title','subtitle','categories','author_name','image_url','description']]
books_export.to_csv('books.csv', index=False, sep='|')
books_export.head(100).to_csv('books_sample.csv', index=False, sep='|')

In [177]:
len(books_export)

4533

In [178]:
reviews_df = pd.read_csv('kindle_reviews_seed.csv')

In [179]:
reviews_df.head(100).to_csv('reviews_sample.csv', index=False)

In [180]:
reviews_df['user_id'].unique().__len__()

2709

In [181]:
reviews_df['item_id'].unique().__len__()

4533

In [182]:
reviews_df.head()

,user_id,item_id,rating,timestamp
0,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B09J8JSJL5,5.0,1671887532659
1,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B08HN77HSL,2.0,1657888532986
2,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B08M6791DZ,5.0,1647002032110
3,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B079ZYWJJ8,5.0,1632326320291
4,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B07WHCL96K,5.0,1601307243310


In [183]:
len(reviews_df)

111005

## Generar matriz de reviews

In [188]:
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

In [185]:
reviews_df = pd.read_csv('kindle_reviews_seed.csv')
reviews_df.head()

,user_id,item_id,rating,timestamp
0,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B09J8JSJL5,5.0,1671887532659
1,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B08HN77HSL,2.0,1657888532986
2,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B08M6791DZ,5.0,1647002032110
3,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B079ZYWJJ8,5.0,1632326320291
4,AHAMHLUWZTVCHUQLQNDZZAAX4KGA,B07WHCL96K,5.0,1601307243310


In [237]:
meta_df = meta_df.set_index('item_id')

In [250]:
reviews_df['user_id'] = reviews_df['user_id'].astype("category")
reviews_df['item_id'] = reviews_df['item_id'].astype("category")

idx_to_userid = dict(enumerate(reviews_df['user_id'].cat.categories))
idx_to_bookid = dict(enumerate(reviews_df['item_id'].cat.categories))

userid_to_idx = { v : k for k, v in idx_to_userid.items() }
bookid_to_idx = { v : k for k, v in idx_to_bookid.items() }

def map_rating_to_confidence(rating):
    if rating >= 4:
        return 1.0
    elif rating == 3:
        return 0.5
    else:
        return 0.0

user_item_matrix = csr_matrix(
    (reviews_df['rating'].astype(float).apply(map_rating_to_confidence), (user_indices, book_indices)),
    shape=(len(reviews_df['user_id'].cat.categories), len(reviews_df['item_id'].cat.categories))
)

In [251]:
user_item_matrix.shape

(2709, 4533)

In [252]:
model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20)
model.fit(user_item_matrix)

  0%|          | 0/20 [00:00<?, ?it/s]

In [253]:
print(idx_to_userid[0])
print(userid_to_idx[idx_to_userid[0]])
print(userid_to_idx[idx_to_userid[0]] == 0)

AE22PJ54OVIRX3I6KSLMPRHPHA4A
0
True


In [254]:
# Get recommendations for the a single user
userid = 0
ids, scores = model.recommend(userid, user_item_matrix[userid], N=10, filter_already_liked_items=False)
book_ids = [idx_to_bookid[id] for id in ids]
print("Recommended book IDs for user {}: {}".format(idx_to_userid[userid], book_ids))

Recommended book IDs for user AE22PJ54OVIRX3I6KSLMPRHPHA4A: ['B00K2EOONI', 'B00BVFM04C', 'B00NAJZDIM', 'B00LXRJICK', 'B00LWE0QZM', 'B0186N2W9Y', 'B007FG9LIE', 'B00TEFTA80', 'B01MRXR9S2', 'B00PM995TG']


In [255]:
meta_df.loc[book_ids]

,title,subtitle,description,categories,images,author,image_url,author_name
item_id,,,,,,,,
B00K2EOONI,My Sister's Grave (Tracy Crosswhite Book 1),Kindle Edition,"[Review, A Goodreads Best Book of the Month, “...","[ 'Mystery, Thriller & Suspense']",[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/61mGjcb5IB...,Robert Dugoni
B00BVFM04C,Breakthrough,Kindle Edition,"[Review, '""If you like Clive Cussler you will ...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51NJsE4WKL...,Michael C. Grumley
B00NAJZDIM,Wreckage,Kindle Edition,"[From the Publisher, 'You know that chill you ...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/41svUp30fP...,Emily Bleeker
B00LXRJICK,Leap (Breakthrough Book 2),NaN,"[From the Author, 'THE COMPLETE BREAKTHROUGH S...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/518NkQMh-X...,Michael C. Grumley
B00LWE0QZM,The Dead Key,NaN,"[From Publishers Weekly, 'This superb novel te...","[ 'Mystery, Thriller & Suspense']",[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/510dmgue3L...,D. M. Pulley
B0186N2W9Y,Catalyst (Breakthrough Book 3),Kindle Edition,"[From the Author, 'THE COMPLETE BREAKTHROUGH S...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51r2eWaUIa...,Michael C. Grumley
B007FG9LIE,"Pines (The Wayward Pines Trilogy, Book 1)",NaN,"[Amazon.com Review, Blake Crouch on How the Te...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/41TiX1qlqj...,Blake Crouch
B00TEFTA80,Her Final Breath (Tracy Crosswhite Book 2),Kindle Edition,"[Review, 'Amazon Best Book of the Month, Myste...","[ 'Mystery, Thriller & Suspense']",[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51mcyr3a8f...,Robert Dugoni
B01MRXR9S2,Ripple (Breakthrough Book 4),Kindle Edition,"[From the Author, 'THE COMPLETE BREAKTHROUGH S...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/510O3luceh...,Michael C. Grumley


In [256]:
import numpy as np
import pandas as pd
pd.DataFrame({"Book": meta_df.loc[book_ids]['title'], "score": scores, "already_liked": np.in1d(ids, user_item_matrix[userid].indices)})

,Book,score,already_liked
item_id,,,
B00K2EOONI,My Sister's Grave (Tracy Crosswhite Book 1),0.951658,True
B00BVFM04C,Breakthrough,0.695341,True
B00NAJZDIM,Wreckage,0.621723,True
B00LXRJICK,Leap (Breakthrough Book 2),0.561169,True
B00LWE0QZM,The Dead Key,0.520448,False
B0186N2W9Y,Catalyst (Breakthrough Book 3),0.419310,True
B007FG9LIE,"Pines (The Wayward Pines Trilogy, Book 1)",0.371441,True
B00TEFTA80,Her Final Breath (Tracy Crosswhite Book 2),0.361116,False
B01MRXR9S2,Ripple (Breakthrough Book 4),0.328329,False


In [257]:
bookid_to_idx

{'B000FA5PZU': 0,
 'B000FA5QMM': 1,
 'B000FA5SMK': 2,
 'B000FA5SPW': 3,
 'B000FA5STS': 4,
 'B000FA5SUM': 5,
 'B000FA64R8': 6,
 'B000FA670W': 7,
 'B000FBFMB6': 8,
 'B000FBFMFM': 9,
 'B000FBFMFW': 10,
 'B000FBFMG6': 11,
 'B000FBFMZ2': 12,
 'B000FBFMZM': 13,
 'B000FBFN1U': 14,
 'B000FBFNME': 15,
 'B000FBFNT2': 16,
 'B000FBJAEQ': 17,
 'B000FBJCJE': 18,
 'B000FBJCK8': 19,
 'B000FBJDF2': 20,
 'B000FBJFSC': 21,
 'B000FBJFSM': 22,
 'B000FBJGEK': 23,
 'B000FBJHKI': 24,
 'B000FBJHZS': 25,
 'B000FC0PDA': 26,
 'B000FC0QBQ': 27,
 'B000FC0QHA': 28,
 'B000FC0SH8': 29,
 'B000FC0SJ6': 30,
 'B000FC0STQ': 31,
 'B000FC0VVQ': 32,
 'B000FC0ZIA': 33,
 'B000FC10S4': 34,
 'B000FC11A6': 35,
 'B000FC128M': 36,
 'B000FC130E': 37,
 'B000FC13Y0': 38,
 'B000FC146C': 39,
 'B000FC14JY': 40,
 'B000FC1AGG': 41,
 'B000FC1BNI': 42,
 'B000FC1HBY': 43,
 'B000FC1IRM': 44,
 'B000FC1KKC': 45,
 'B000FC1MBO': 46,
 'B000FC1MCS': 47,
 'B000FC1PWA': 48,
 'B000FC1RSC': 49,
 'B000FC1SCW': 50,
 'B000FC2L1E': 51,
 'B000FC2L1O': 52,
 'B

In [258]:
meta_df.iloc[0]['title']

'ROLLING THUNDER: An Historical Novel of War and Politics (Wings of War Book 1)'

In [259]:
ids, scores= model.similar_items(0)

# display the results using pandas for nicer formatting
pd.DataFrame({"Book": meta_df.iloc[ids]['title'], "score": scores})

,Book,score
item_id,,
B0019LNUVA,ROLLING THUNDER: An Historical Novel of War an...,1.000000
B01GY95VZE,Weightless: A Small Town Romance,0.785351
B00FJJN0WC,Friends Without Benefits: An Unrequited Love R...,0.746337
B00DQGMT6S,Goliath (A Ryan Mitchell Thriller Book 1),0.738551
B00KVI9DP4,The Reluctant Midwife: A Hope River Novel,0.712181
B00A2C53M6,The Astronaut Wives Club: A True Story,0.708064
B07MRJG5C9,A Place Without you,0.701620
B000FCK3C8,"The Angel Experiment (Maximum Ride, Book 1): A...",0.699391
B003YL4LYI,"A Dance with Dragons (A Song of Ice and Fire, ...",0.689331


In [260]:
meta_df[meta_df['author_name'] == 'Brandon Sanderson'].head()

,title,subtitle,description,categories,images,author,image_url,author_name
item_id,,,,,,,,
B00DA6YEKS,"Words of Radiance (The Stormlight Archive, Boo...",Kindle Edition,"[From the Artist, Brandon Sanderson; Read by M...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51jVA6k9AU...,Brandon Sanderson
B002KYHZHA,Warbreaker,Kindle Edition,"[Review, '""Not only has Sanderson drawn a fres...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51yNKw2vXS...,Brandon Sanderson
B00VZZ085G,Calamity (The Reckoners Book 3),Kindle Edition,"[Review, 'Praise for the Reckoners series:', #...",[ 'Children\'s eBooks'],[],{'avatar': 'https://m.media-amazon.com/images/...,,Brandon Sanderson
B002GYI9C4,Mistborn: The Final Empire,Kindle Edition,"[Review, '""Elantris', 'is the finest novel of ...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51G0zeHL2F...,Brandon Sanderson
B0826NKZHR,Rhythm of War: Book Four of The Stormlight Arc...,Kindle Edition,"[Review, Praise for the Stormlight Archive, '“...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...,{'avatar': 'https://m.media-amazon.com/images/...,https://m.media-amazon.com/images/I/51qrprjSJI...,Brandon Sanderson


In [261]:
mistborn_id = bookid_to_idx['B002GYI9C4']

ids, scores= model.similar_items(mistborn_id)
book_ids = [idx_to_bookid[id] for id in ids]

# display the results using pandas for nicer formatting
pd.DataFrame({"Book": meta_df.loc[book_ids]['title'], "score": scores})

,Book,score
item_id,,
B002GYI9C4,Mistborn: The Final Empire,1.000000
B000UZQI0Q,The Well of Ascension: Book Two of Mistborn,0.914236
B000FBJCK8,Eragon: Book I (The Inheritance Cycle 1),0.905808
B002LC8HF0,The Hero of Ages: Book Three of Mistborn,0.896956
B003P2WO5E,"The Way of Kings (The Stormlight Archive, Book 1)",0.890302
B00HBQWGYO,Half a King (Shattered Sea Book 1),0.880805
B00H25FCNG,The Broken Eye (Lightbringer Book 3),0.879732
B003JTHY76,The Black Prism (Lightbringer Book 1),0.872897
B00ARHAAZ6,Steelheart (The Reckoners Book 1),0.858499


## Entrenar modelo con Surprise

In [41]:
from surprise import Dataset, SVD, Reader, accuracy
from surprise.model_selection import train_test_split

In [29]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(reviews_df[['user_id', 'item_id', 'rating']], reader)

In [48]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [49]:
algo = SVD()
algo.fit(trainset)


In [50]:
predictions = algo.test(testset)

In [51]:
accuracy.rmse(predictions)

RMSE: 0.7607


0.7607130593338862

In [52]:
df_test = pd.DataFrame(testset, columns=['user_id', 'item_id', 'rating_real'])
df_test.to_csv('testset_oraculo.csv', index=False)

In [35]:
import joblib

In [55]:
joblib.dump(algo, 'svd_model.pkl')

['svd_model.pkl']

## Generar embeddings de los libros

In [6]:
from sentence_transformers import SentenceTransformer

In [2]:
books_df = pd.read_csv('kindle_meta_cleaned.csv')
books_df.head()

,item_id,title,subtitle,description,categories,images
0,B094XQFL8P,Runaway (Empire High Book 5),Kindle Edition,"[About the Author, 'Ivy Smoak is the internati...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
1,B075WQK8ZJ,Unsheltered: A Novel,Kindle Edition,"[Amazon.com Review, 'An Amazon Best Book of Oc...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
2,B09JS55FW3,Reaper (Cradle Book 10),Kindle Edition,"[About the Author, Will Wight, 'lives in Flori...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...
3,B01JJ5CURW,To-Do List Formula: A Stress-Free Guide To Cre...,Kindle Edition,[],[ Business & Money],[{'large': 'https://m.media-amazon.com/images/...
4,B00KQBTC3O,Shopping for a Billionaire 1 (Shopping for a B...,Kindle Edition,"[From the Author, 'Please sign up for my New R...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...


In [15]:
# Título y categorías primeros, luego descripción. Esto es porque la descripción suele ser más larga y menos importante, por lo que el modelo la puede truncar por el número máximo de tokens.
books_df['complete_text'] = books_df['title'].fillna('') + books_df['categories'].fillna('') + books_df['description'].fillna('')

In [16]:
books_df.iloc[0].complete_text

"Runaway (Empire High Book 5)[  Literature & Fiction][About the Author, 'Ivy Smoak is the international bestselling author of The Hunted Series. Her books have sold over 1 million copies worldwide, and her latest release, Empire High Betrayal, hit #4 in the entire Kindle store. When she\\'s not writing, you can find Ivy binge watching too many TV shows, taking long walks, playing outside, and generally refusing to act like an adult. She lives with her husband in Delaware.', --This text refers to the, hardcover, edition.]"

Ver largo del texto

In [17]:
books_df['text_length'] = books_df['complete_text'].apply(lambda x: len(x.split()))

In [18]:
books_df[['item_id', 'complete_text', 'text_length']].head()

,item_id,complete_text,text_length
0,B094XQFL8P,Runaway (Empire High Book 5)[ Literature & Fi...,85
1,B075WQK8ZJ,Unsheltered: A Novel[ Literature & Fiction][A...,1587
2,B09JS55FW3,Reaper (Cradle Book 10)[ Science Fiction & Fa...,96
3,B01JJ5CURW,To-Do List Formula: A Stress-Free Guide To Cre...,15
4,B00KQBTC3O,Shopping for a Billionaire 1 (Shopping for a B...,138


In [20]:
books_df['text_length'].describe()

count     4601.000000
mean      1314.534232
std       2290.880745
min          3.000000
25%        135.000000
50%        405.000000
75%       1514.000000
max      43958.000000
Name: text_length, dtype: float64

In [21]:
nlp_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
# Probar el modelo
sentence = "This is an example sentence"
embedding = nlp_model.encode(sentence)
print(embedding)

[ 0.18845688  0.17425634  0.0544778   0.29051757  0.16766414 -0.04720675
  0.6455802   0.1598089   0.22689255 -0.03089058  0.2558835  -0.05258775
 -0.2261015  -0.05710635  0.13042626  0.12495356  0.31749612  0.19444402
 -0.5863256  -0.01258597  0.6099092   0.16432735  0.03331115 -0.27383068
 -0.28975752 -0.21119711 -0.02261387 -0.17035921  0.1615901   0.06082748
 -0.24162416  0.18579195  0.4274097   0.19295172 -0.07234465  0.16611089
  0.10442799  0.20477234  0.21116723  0.19973989 -0.09408262 -0.1738367
  0.06427368  0.28025475 -0.29530594  0.0620954   0.10427695 -0.02364418
  0.1291318  -0.12617467 -0.17899004  0.0370057  -0.6125061   0.05029812
  0.17730357  0.22494122  0.17386065 -0.03840291 -0.2128681   0.2584925
 -0.12101632  0.30971512 -0.41966376  0.00907662  0.14188907 -0.30556947
  0.17621145 -0.07087357 -0.62033135  0.6770835   0.01723751  0.18405128
 -0.1678578   0.20452645 -0.1477028  -0.06175363  0.63017434  0.11120182
  0.0515306   0.15927422 -0.05370894  0.05350953  0.1

In [26]:
nlp_model.max_seq_length = 256

In [32]:
text_list = books_df['complete_text'].tolist()
vectors = nlp_model.encode(text_list, show_progress_bar=True)

Batches:   0%|          | 0/144 [00:00<?, ?it/s]

In [33]:
embeddings_dict = {
    item_id: vector 
    for item_id, vector in zip(books_df['item_id'], vectors)
}

In [36]:
joblib.dump(embeddings_dict, 'book_embeddings.pkl')

['book_embeddings.pkl']